# 07 - Run The Full System Eval

Runs the full offline benchmark path: route, retrieve, relevance proxy, optional fallback decision, answer extraction, and aggregate metrics. It supports both base and challenging datasets and records stage latency.


## Learning Goal

Combine the earlier component evals into one offline benchmark: route, retrieve, check context, decide fallback, produce an answer, and measure latency. This lab teaches students when to move from isolated component checks to an end-to-end system view.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> fallback eval -> answer quality -> full benchmark -> ablation.

This is the first full-system notebook. It is useful for seeing total behavior, but students should still use earlier labs to debug which component caused a failure.

## Related AI Evals Concepts

- Traces For Evals: end-to-end rows preserve stage outputs for debugging.
- Deploy Evals: benchmark summaries and exported artifacts are the first step toward repeated evaluation.
- Types Of Automated Evals: combine deterministic metrics, proxy decisions, and latency measurements.
- AI Eval Mistakes: aggregate system metrics can hide component-level failures.


In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag')

In [11]:
import asyncio
import importlib
import os
import json
import time
from typing import Any

import chromadb
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import agentic_rag.ingestion as ingestion_module
import agentic_rag.retrievers as retrievers_module
import agentic_rag.router as router_module
from agentic_rag.constants import SourceType
from agentic_rag.evaluation import parse_doc_ids, score_retrieval

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections
from agentic_rag.llm import OpenAITextGenerator
from agentic_rag.retrievers import ChromaRetriever
from agentic_rag.router import QueryRouter
from agentic_rag.settings import Settings
from agentic_rag.telemetry import configure_tracing, start_span

pd.set_option("display.max_colwidth", 180)


In [12]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "07_full_agentic_rag_benchmark.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/otel_traces/07_full_agentic_rag_benchmark.jsonl')

## Load Full-Benchmark Test Sets


In [13]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


,dataset,query,expected_source_type
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA


In [14]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


,collection,exists,count
0,medical_qna,True,16407
1,medical_device_manual,True,2694


The CSV `expected_doc_ids` column stores local row numbers from the generated eval file, while Chroma returns collection document IDs such as `qna-7354` and `device-346`. Full-benchmark retrieval metrics compare IDs by exact string match, so this notebook resolves each gold row back to the Chroma document ID before scoring.


In [15]:
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}


def normalize_text(value: object) -> str:
    return " ".join(str(value or "").lower().split())


qna_id_by_question_answer = {
    (normalize_text(row["Question"]), normalize_text(row["Answer"])): f"qna-{row_index}"
    for row_index, row in qna_df.reset_index(drop=True).iterrows()
}
qna_ids_by_question = qna_df.reset_index(drop=True).groupby(
    qna_df["Question"].map(normalize_text)
).apply(lambda rows: [f"qna-{row_index}" for row_index in rows.index]).to_dict()

device_answer_columns = ["Indications_for_Use", "Contraindications", "Patient_Population"]
device_lookup_rows = []
for row_index, row in device_df.reset_index(drop=True).iterrows():
    for column in device_answer_columns:
        if column in row and not pd.isna(row[column]):
            device_lookup_rows.append(
                {
                    "doc_id": f"device-{row_index}",
                    "answer": normalize_text(row[column]),
                    "device_name": normalize_text(row["Device_Name"]),
                    "model_number": normalize_text(row["Model_Number"]),
                }
            )


def local_source_from_value(value: str) -> SourceType | None:
    try:
        source = SourceType(value)
    except ValueError:
        return None
    return source if source.value in LOCAL_SOURCES else None


def resolve_expected_doc_ids(row: pd.Series) -> list[str]:
    source = local_source_from_value(str(row["expected_source_type"]))
    query = normalize_text(row["query"])
    expected_answer = normalize_text(row.get("expected_answer", ""))

    if source == SourceType.RETRIEVE_QNA:
        exact_match = qna_id_by_question_answer.get((query, expected_answer))
        if exact_match:
            return [exact_match]
        return qna_ids_by_question.get(query, [])

    if source == SourceType.RETRIEVE_DEVICE:
        candidates = [item for item in device_lookup_rows if item["answer"] == expected_answer]
        model_matches = [item for item in candidates if item["model_number"] and item["model_number"] in query]
        if model_matches:
            return sorted({item["doc_id"] for item in model_matches})
        named_matches = [item for item in candidates if item["device_name"] and item["device_name"] in query]
        if named_matches:
            return sorted({item["doc_id"] for item in named_matches})
        return sorted({item["doc_id"] for item in candidates})

    return []


def expected_ids_from_row(row: pd.Series) -> list[str]:
    resolved_ids = resolve_expected_doc_ids(row)
    return resolved_ids or parse_doc_ids(row.get("expected_doc_ids", ""))


## Define End-To-End Eval Stages


In [16]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

async def full_one(row: pd.Series, top_k: int = 3) -> dict[str, Any]:
    started = time.perf_counter()
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    expected_ids = expected_ids_from_row(row)
    router = QueryRouter(mode="heuristic")

    with start_span("notebook.full_benchmark.row", dataset=str(row["dataset"]), query_length=len(query)):
        route_start = time.perf_counter()
        predicted_source = (await router.route(query)).value
        route_ms = (time.perf_counter() - route_start) * 1000

        retrieval_ms = 0.0
        docs = []
        if is_local_source(predicted_source):
            retrieval_start = time.perf_counter()
            retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
            docs = await retriever.retrieve(SourceType(predicted_source), query)
            retrieval_ms = (time.perf_counter() - retrieval_start) * 1000

        retrieved_ids = [doc.doc_id for doc in docs]
        retrieval_score = score_retrieval(retrieved_ids, expected_ids)
        context = "\n".join(doc.text for doc in docs)
        context_relevant = retrieval_score.hit_at_k == 1 if expected_ids else (lexical_score(query, context) >= 0.2 if context else predicted_source == SourceType.WEB_SEARCH.value)
        fallback_triggered = is_local_source(predicted_source) and not context_relevant
        final_source = SourceType.WEB_SEARCH.value if fallback_triggered else predicted_source
        answer = preview_text(docs[0].text, max_words=50) if docs else ""

        return {
            "dataset": row["dataset"],
            "query": query,
            "expected_source_type": expected_source,
            "predicted_source_type": predicted_source,
            "route_correct": predicted_source == expected_source,
            "retrieved_doc_ids": "|".join(retrieved_ids),
            "expected_doc_ids": "|".join(expected_ids),
            "hit_at_k": retrieval_score.hit_at_k if expected_ids else None,
            "recall_at_k": retrieval_score.recall_at_k if expected_ids else None,
            "context_relevant": context_relevant,
            "fallback_triggered": fallback_triggered,
            "final_source_type": final_source,
            "actual_answer": answer,
            "lexical_answer_relevance": lexical_score(query, answer) if answer else None,
            "route_latency_ms": route_ms,
            "retrieval_latency_ms": retrieval_ms,
            "total_latency_ms": (time.perf_counter() - started) * 1000,
            "metric_status": "scored" if expected_ids else ("qualitative_only_no_gold_doc_ids" if is_local_source(expected_source) else "skipped_web_source"),
        }


async def full_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        rows.append(await full_one(row))
    return pd.DataFrame(rows)


def summarize_full(results: pd.DataFrame) -> pd.DataFrame:
    scored = results[results["metric_status"] == "scored"]
    return pd.DataFrame([
        {
            "examples": len(results),
            "router_accuracy": float(results["route_correct"].mean()),
            "hit_at_k": float(scored["hit_at_k"].mean()) if not scored.empty else None,
            "recall_at_k": float(scored["recall_at_k"].mean()) if not scored.empty else None,
            "fallback_rate": float(results["fallback_triggered"].mean()),
            "mean_total_latency_ms": float(results["total_latency_ms"].mean()),
        }
    ])


## Run The Full Offline System Eval


In [17]:
base_full_results = await full_benchmark(base_df)
challenge_full_results = await full_benchmark(challenge_df)

base_full_results.head()


,dataset,query,expected_source_type,predicted_source_type,route_correct,retrieved_doc_ids,expected_doc_ids,hit_at_k,recall_at_k,context_relevant,fallback_triggered,final_source_type,actual_answer,lexical_answer_relevance,route_latency_ms,retrieval_latency_ms,total_latency_ms,metric_status
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA,Retrieve_QnA,True,qna-7354|qna-10769|qna-10404,qna-7354,1.0,1.0,True,False,Retrieve_QnA,Question: How many people are affected by X-linked chondrodysplasia punctata 1 ? Answer: The prevalence of X-linked chondrodysplasia punctata 1 is unknown. Several dozen affect...,1.000000,0.316541,2.718708,3.134583,scored
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA,Retrieve_QnA,True,qna-12125|qna-2142|qna-13126,qna-6837,0.0,0.0,False,True,Web_Search,Question: What are the treatments for Tubular aggregate myopathy ? Answer: How might tubular aggregate myopathy be treated?,0.500000,0.052084,2.092541,2.215542,scored
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA,Retrieve_QnA,True,qna-8175|qna-6615|qna-10705,qna-8175,1.0,1.0,True,False,Retrieve_QnA,Question: What are the genetic changes related to Ellis-van Creveld syndrome ? Answer: Ellis-van Creveld syndrome can be caused by mutations in the EVC or EVC2 gene. Little is ...,1.000000,0.052208,2.303625,2.447709,scored
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA,Retrieve_QnA,True,qna-15015|qna-13143|qna-12661,qna-14533,0.0,0.0,False,True,Web_Search,Question: What are the symptoms of Bell's palsy ? Answer: What are the symptoms of Bell's palsy?,0.333333,0.046375,1.771250,1.882084,scored
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA,Retrieve_QnA,True,qna-15453|qna-16387|qna-12450,qna-9958,0.0,0.0,False,True,Web_Search,"Question: What is (are) Cri du chat syndrome ? Answer: Cri du chat syndrome, also known as 5p- (5p minus) syndrome or cat cry syndrome, is a genetic condition that is caused by...",0.750000,0.053917,1.784000,1.912834,scored


In [18]:
base_full_summary = summarize_full(base_full_results)
base_full_summary.insert(0, "dataset", "base")
challenge_full_summary = summarize_full(challenge_full_results)
challenge_full_summary.insert(0, "dataset", "challenging")

full_summary = pd.concat([base_full_summary, challenge_full_summary], ignore_index=True)
full_summary


,dataset,examples,router_accuracy,hit_at_k,recall_at_k,fallback_rate,mean_total_latency_ms
0,base,27,1.000000,0.291667,0.291667,0.629630,1.897758
1,challenging,15,0.733333,None,None,0.133333,1.558308


In [19]:
challenge_full_results.loc[
    ~challenge_full_results["route_correct"],
    ["query", "expected_source_type", "predicted_source_type", "context_relevant", "fallback_triggered", "final_source_type"],
]


,query,expected_source_type,predicted_source_type,context_relevant,fallback_triggered,final_source_type
3,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,Retrieve_QnA,False,True,Web_Search
9,"If the retrieved Q&A answer about dystonia is irrelevant, should the graph fall back to Tavily?",Web_Search,Retrieve_QnA,True,False,Retrieve_QnA
11,Are there recent safety alerts for the manufacturer Abbott's electrosurgical units?,Web_Search,Retrieve_Device,True,False,Retrieve_Device
14,"Who is at risk for LCM, and has there been a recent outbreak near Athens?",Web_Search,Retrieve_QnA,False,True,Web_Search


## Pay Attention To

- End-to-end metrics are useful summaries, but they are not enough for debugging.
- Preserve stage-level columns so a failed row can be traced back to routing, retrieval, fallback, or answering.
- Latency is part of product quality, especially when adding LLM judges or web fallback.
- Full benchmarks are most useful after component evals already exist.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against both the base and challenging router datasets. It reads `OPENAI_API_KEY` from the active environment only; it does not load `.env`.


In [20]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_router_results = pd.DataFrame()
llm_router_summary = pd.DataFrame()


def load_router_eval_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Query" in df.columns:
        df = df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})
    df = df.copy()
    df["dataset"] = dataset_name
    df["expected_source_type"] = df["expected_source_type"].astype(str)
    return df


async def evaluate_llm_router_dataset(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(str(query)) for query in df["query"].tolist()))
    results = df.copy()
    results["router_mode"] = "llm"
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    keep_columns = [
        "dataset",
        "router_mode",
        "query",
        "expected_source_type",
        "predicted_source_type",
        "route_correct",
        "category",
        "rationale",
    ]
    return results[[column for column in keep_columns if column in results.columns]]


def summarize_llm_router_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    return results.groupby(["dataset", "router_mode"], dropna=False).agg(
        examples=("query", "count"),
        accuracy=("route_correct", "mean"),
        failures=("route_correct", lambda values: int((~values).sum())),
    ).reset_index()


if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    llm_router = QueryRouter(
        generator=OpenAITextGenerator(api_key=api_key, model=LLM_ROUTER_MODEL, timeout=LLM_ROUTER_TIMEOUT),
        mode="llm",
    )
    llm_router_base_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/evaluation_dataset.csv", "base")
    llm_router_challenge_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv", "challenging")
    llm_router_results = pd.concat(
        [
            await evaluate_llm_router_dataset(llm_router, llm_router_base_df),
            await evaluate_llm_router_dataset(llm_router, llm_router_challenge_df),
        ],
        ignore_index=True,
    )
    llm_router_summary = summarize_llm_router_results(llm_router_results)

llm_router_summary


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.0,0
1,challenging,llm,15,0.8,3


In [21]:
if not llm_router_results.empty:
    display(llm_router_summary)
    display(pd.crosstab(
        [llm_router_results["dataset"], llm_router_results["expected_source_type"]],
        llm_router_results["predicted_source_type"],
        dropna=False,
    ))
    display(llm_router_results.loc[~llm_router_results["route_correct"]])


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.0,0
1,challenging,llm,15,0.8,3


predicted_source_type             Retrieve_Device  Retrieve_QnA  Web_Search
dataset     expected_source_type                                           
base        Retrieve_Device                    12             0           0
            Retrieve_QnA                        0            12           0
            Web_Search                          0             0           3
challenging Retrieve_Device                     5             2           0
            Retrieve_QnA                        0             1           0
            Web_Search                          0             1           6

,dataset,router_mode,query,expected_source_type,predicted_source_type,route_correct,category,rationale
27,challenging,llm,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Retrieve_QnA,False,device_plus_general_medical,"Mentions general risks, but the answer depends on a device manual contraindication."
28,challenging,llm,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,Retrieve_QnA,False,explicit_source_meta_question,Asks about source selection and current outbreak updates rather than a stable disease answer.
33,challenging,llm,What is Fraser syndrome and can any surgical robot in our manuals be used for related procedures?,Retrieve_Device,Retrieve_QnA,False,mixed_qna_device,"Contains a disease definition, but asks whether manuals contain a suitable device."


## Export Full-Benchmark Artifacts


In [22]:
full_results = pd.concat([base_full_results, challenge_full_results], ignore_index=True)
results_path = PROJECT_ROOT / "output/full_agentic_rag_benchmark_results.csv"
summary_path = PROJECT_ROOT / "output/full_agentic_rag_benchmark_summary.csv"
full_results.to_csv(results_path, index=False)
full_summary.to_csv(summary_path, index=False)

results_path, summary_path


(PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/full_agentic_rag_benchmark_results.csv'),
 PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/full_agentic_rag_benchmark_summary.csv'))